# 05 — Fundamental & Quality Alpha Research

Fundamental alphas use accounting data (balance sheet, income statement, cash flow)
to find mispriced stocks. They tend to have **lower turnover** than price-based
alphas — fundamentals update quarterly — which benefits the fitness formula.

## Alpha ideas by theme

**Value**
- Book-to-market: `group_rank(book_to_price, subindustry)` — buy cheap within sector
- Earnings yield: `group_rank(earnings_per_share / close, subindustry)`

**Quality**
- ROE trend: `group_rank(ts_delta(roe, 4), subindustry)` — buy improving profitability
- Gross margin momentum: `ts_delta(gross_margin, 4)` — improving margins → outperformance
- Asset growth penalty: `-rank(ts_delta(total_assets, 4))` — firms that grow assets excessively underperform

**Profitability**
- Operating leverage: `rank(operating_income / revenue)` 
- Cash conversion: `rank(cash_from_operations / net_income)` — quality earnings vs paper earnings

## Key settings for fundamental alphas

- `decay=10–20` — fundamental signals update slowly; smooth the signal
- `neutralization='Subindustry'` — ALWAYS neutralize fundamentals within sector to avoid sector bets
- `delay=1` — fundamentals are reported with a delay; use delay=1 to avoid look-ahead bias

In [1]:
import sys
from pathlib import Path
import pandas as pd
from tqdm.notebook import tqdm

repo_root = Path.cwd().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from wq.client import get_client, simulate
from wq.alpha import AlphaConfig, fundamental_alphas, FUNDAMENTAL_TEMPLATES, expand_templates
from wq.utils import load_datafields, search_fields, load_results, top_alphas, parse_result

client = get_client()

Authenticated as jayesh.s@alumni.iitgn.ac.in


## Step 1: Find relevant fundamental fields from the catalog

In [2]:
catalog = load_datafields()

# Look for value-related fields
value_fields = search_fields(catalog, keyword='book', category='fundamental')
print(f'Book-related fields: {len(value_fields)}')
value_fields.head(10)

Book-related fields: 3


,id,description,region,delay,universe,type,datecoverage,coverage,usercount,alphacount,themes,dataset_id,dataset_name,category_id,category_name,subcategory_id,subcategory_name
0,bookvalue_ps,Book Value Per Share,USA,1,TOP3000,MATRIX,1.0,0.5000,2640,9845,[],fundamental6,Company Fundamental Data for Equity,fundamental,Fundamental,fundamental-fundamental-data,Fundamental Data
1,fnd6_newa1v1300_bkvlps,Book Value Per Share,USA,1,TOP3000,MATRIX,1.0,0.5000,252,948,[],fundamental6,Company Fundamental Data for Equity,fundamental,Fundamental,fundamental-fundamental-data,Fundamental Data
2,fnd2_a_frtandfixturesg,Amount before accumulated depreciation of equi...,USA,1,TOP3000,MATRIX,NaN,0.3549,111,342,[],fundamental2,Report Footnotes,fundamental,Fundamental,fundamental-footnotes,Footnotes


In [3]:
# ROE / profitability fields
quality_fields = search_fields(catalog, keyword='return on equity')
quality_fields2 = search_fields(catalog, keyword='roe')
pd.concat([quality_fields, quality_fields2]).drop_duplicates(subset='id').head(15)

,id,description,region,delay,universe,type,datecoverage,coverage,usercount,alphacount,themes,dataset_id,dataset_name,category_id,category_name,subcategory_id,subcategory_name
0,return_equity,Return on Equity,USA,1,TOP3000,MATRIX,1.0,0.5000,1851,8553,[],fundamental6,Company Fundamental Data for Equity,fundamental,Fundamental,fundamental-fundamental-data,Fundamental Data
1,mdl177_2_garpanalystmodel_qgp_roefcf,Combined rank measuring both Return on Equity ...,USA,1,TOP3000,MATRIX,1.0,0.9059,152,304,[],model77,Analysts' Factor Model,model,Model,model-technical-models,Technical Models
2,mdl177_earningsmomemtummodel_fc_fcfroey1p_alt,Product of trailing 12-month free cash flow yi...,USA,1,TOP3000,MATRIX,1.0,1.0000,57,234,[],model77,Analysts' Factor Model,model,Model,model-technical-models,Technical Models
3,mdl177_2_deepvaluemodel2_dv_currroe,"Current return on equity for the stock, comput...",USA,1,TOP3000,MATRIX,1.0,0.9948,69,177,[],model77,Analysts' Factor Model,model,Model,model-technical-models,Technical Models
4,mdl177_2_managementqualityfactor_indrelcroe_,Stock’s lagged quarterly return on equity minu...,USA,1,TOP3000,MATRIX,1.0,0.9547,37,157,[],model77,Analysts' Factor Model,model,Model,model-technical-models,Technical Models
5,mdl177_earningmomentumfactor_fcfroey1p_alt,Product of trailing twelve-month free cash flo...,USA,1,TOP3000,MATRIX,1.0,0.9654,40,142,[],model77,Analysts' Factor Model,model,Model,model-technical-models,Technical Models
6,mdl177_2_growthanalystmodel_qga_niroe,Return on equity computed using net income (Ne...,USA,1,TOP3000,MATRIX,1.0,0.9940,34,130,[],model77,Analysts' Factor Model,model,Model,model-technical-models,Technical Models
7,mdl177_2_managementqualityfactor_fwdroe,Forward (projected) return on equity,USA,1,TOP3000,MATRIX,1.0,0.9219,43,128,[],model77,Analysts' Factor Model,model,Model,model-technical-models,Technical Models
8,mdl177_relativevaluemodel_roe,Return on Equity; trailing 12-month income bef...,USA,1,TOP3000,MATRIX,1.0,0.9989,35,103,[],model77,Analysts' Factor Model,model,Model,model-technical-models,Technical Models
9,mdl177_2_earningmomentumfactor400_fcfroey1p,Product of trailing 12-month free cash flow yi...,USA,1,TOP3000,MATRIX,1.0,0.9462,42,94,[],model77,Analysts' Factor Model,model,Model,model-technical-models,Technical Models


In [4]:
# Margin-related fields
margin_fields = search_fields(catalog, keyword='margin')
print(f'Margin fields: {len(margin_fields)}')
margin_fields.head(10)

Margin fields: 65


,id,description,region,delay,universe,type,datecoverage,coverage,usercount,alphacount,themes,dataset_id,dataset_name,category_id,category_name,subcategory_id,subcategory_name
0,mdl177_historicalgrowthfactor_chgnpm_alt,Most recent quarterly net profit margin minus ...,USA,1,TOP3000,MATRIX,1.0,0.9621,134,405,[],model77,Analysts' Factor Model,model,Model,model-technical-models,Technical Models
1,mdl177_2_growthanalystmodel_qga_opmarginsales,Measure of linkage between operating margin an...,USA,1,TOP3000,MATRIX,1.0,1.0000,226,369,[],model77,Analysts' Factor Model,model,Model,model-technical-models,Technical Models
2,mdl177_growthanalystmodel_qga_opmarginsales_alt,The relationship between operating margin and ...,USA,1,TOP3000,MATRIX,1.0,1.0000,188,322,[],model77,Analysts' Factor Model,model,Model,model-technical-models,Technical Models
3,mdl177_historicalgrowthfactor_chgnpm,Difference between most recent quarterly net p...,USA,1,TOP3000,MATRIX,1.0,0.9562,65,253,[],model77,Analysts' Factor Model,model,Model,model-technical-models,Technical Models
4,mdl177_fa_yoychggpm,Yearly Change in Gross Profit Margin: Change i...,USA,1,TOP3000,MATRIX,1.0,0.8406,64,207,[],model77,Analysts' Factor Model,model,Model,model-technical-models,Technical Models
5,mdl177_fa_chgnpm,One-year change in net profit margin (most rec...,USA,1,TOP3000,MATRIX,1.0,0.9565,56,205,[],model77,Analysts' Factor Model,model,Model,model-technical-models,Technical Models
6,mdl177_2_managementqualityfactor_nopatmargin,Net operating profit after tax (NOPAT) margin:...,USA,1,TOP3000,MATRIX,1.0,0.9047,50,174,[],model77,Analysts' Factor Model,model,Model,model-technical-models,Technical Models
7,mdl177_2_historicalgrowthfactor_chgnpm,Change in net profit margin from one year ago ...,USA,1,TOP3000,MATRIX,1.0,0.9562,69,152,[],model77,Analysts' Factor Model,model,Model,model-technical-models,Technical Models
8,mdl177_2_historicalgrowthfactor_chgopm,Change in quarterly operating profit margin ov...,USA,1,TOP3000,MATRIX,1.0,0.9384,39,151,[],model77,Analysts' Factor Model,model,Model,model-technical-models,Technical Models
9,mdl177_2_managementqualityfactor_npm,Net profit margin: most recent quarterly net i...,USA,1,TOP3000,MATRIX,1.0,0.9669,95,149,[],model77,Analysts' Factor Model,model,Model,model-technical-models,Technical Models


## Step 2: Define hand-crafted fundamental alphas

After exploring the catalog, we specify specific alphas based on economic intuition.

In [5]:
# Replace field IDs below with actual IDs from your catalog search
# Use: search_fields(catalog, keyword='...').iloc[0]['id']

FUNDAMENTAL_SETTINGS = dict(
    universe='TOP3000',
    region='USA',
    neutralization='Subindustry',
    delay=1,
    decay=10,  # smooth fundamental signals
)

fundamental_configs = [
    # Value: buy cheap stocks within their subindustry
    AlphaConfig(
        expression="group_rank(book_to_price, subindustry)",
        tags=['fundamental', 'value'],
        notes="Book-to-price value factor, subindustry ranked",
        **FUNDAMENTAL_SETTINGS,
    ),
    # Quality: improving ROE
    AlphaConfig(
        expression="group_rank(ts_delta(roe, 4), subindustry)",
        tags=['fundamental', 'quality', 'roe'],
        notes="Stocks with improving ROE over last 4 quarters",
        **FUNDAMENTAL_SETTINGS,
    ),
    # Asset growth penalty
    AlphaConfig(
        expression="-1 * group_rank(ts_delta(total_assets, 4), subindustry)",
        tags=['fundamental', 'quality', 'accruals'],
        notes="Penalise aggressive asset growth (overinvestment)",
        **FUNDAMENTAL_SETTINGS,
    ),
    # Earnings quality: cash earnings vs accruals
    AlphaConfig(
        expression="group_rank(cash_from_operations / (net_income + 1e-6), subindustry)",
        tags=['fundamental', 'quality', 'accruals'],
        notes="Cash conversion quality — real earnings vs paper earnings",
        **FUNDAMENTAL_SETTINGS,
    ),
]

print(f'{len(fundamental_configs)} fundamental alphas defined')
for c in fundamental_configs:
    print(f'  {c.expression}')

4 fundamental alphas defined
  group_rank(book_to_price, subindustry)
  group_rank(ts_delta(roe, 4), subindustry)
  -1 * group_rank(ts_delta(total_assets, 4), subindustry)
  group_rank(cash_from_operations / (net_income + 1e-6), subindustry)


## Step 3: Run a systematic sweep over fundamental fields

In [6]:
# Identify the top-N most-used fundamental fields and auto-generate alphas
count_col = next((c for c in catalog.columns if 'alphacount' in c.lower()), None)
cat_col = next((c for c in catalog.columns if 'category' in c.lower() and 'sub' not in c.lower()), None)
id_col = next((c for c in catalog.columns if c.lower() == 'id'), None)

if cat_col and count_col and id_col:
    top_fundamental = (
        catalog[catalog[cat_col].str.lower().str.contains('fundamental', na=False)]
        .nlargest(10, count_col)[id_col].tolist()
    )
    print('Top fundamental fields (by alphaCount):', top_fundamental)

    sweep_configs = []
    for field_id in top_fundamental:
        for expr in expand_templates(FUNDAMENTAL_TEMPLATES, field_id, window=4):
            sweep_configs.append(AlphaConfig(
                expression=expr,
                tags=['fundamental', 'sweep', field_id],
                **FUNDAMENTAL_SETTINGS,
            ))
    print(f'{len(sweep_configs)} sweep alphas generated')
else:
    sweep_configs = []
    print('Could not identify fundamental fields — check catalog column names above')

Top fundamental fields (by alphaCount): ['assets', 'liabilities', 'operating_income', 'enterprise_value', 'sales', 'debt', 'capex', 'equity', 'ebitda', 'ebit']
50 sweep alphas generated


In [7]:
# Combine hand-crafted + sweep
all_fundamental = fundamental_configs + sweep_configs

# De-duplicate by fingerprint
existing = load_results()
done = set(existing['fingerprint'].dropna()) if not existing.empty else set()
to_run = [c for c in all_fundamental if c.fingerprint() not in done]
print(f'Running {len(to_run)} fundamental alphas')

Running 54 fundamental alphas


In [8]:
for config in tqdm(to_run, desc='Fundamental sweep'):
    try:
        result = simulate(client, config, save=True)
        m = parse_result(result)
        print(f'  sharpe={m["is_sharpe"]}  fitness={m["is_fitness"]}  {config.expression[:60]}')
    except Exception as e:
        print(f'  ERROR ({config.expression[:50]}): {e}')

Fundamental sweep:   0%|          | 0/54 [00:00<?, ?it/s]

Submitting: group_rank(book_to_price, subindustry)
  sim_id=2RRquS5Bp5jMaX3pMtrzCVr — polling every 10s...
  status=ERROR
  sharpe=None  fitness=None  group_rank(book_to_price, subindustry)
Submitting: group_rank(ts_delta(roe, 4), subindustry)
  sim_id=12NVqY9JB4gM9NMKNeXSlhz — polling every 10s...
  status=ERROR
  sharpe=None  fitness=None  group_rank(ts_delta(roe, 4), subindustry)
Submitting: -1 * group_rank(ts_delta(total_assets, 4), subindustry)
  sim_id=gTcAf2uP4AJabta4lGt5Fw — polling every 10s...
  status=ERROR
  sharpe=None  fitness=None  -1 * group_rank(ts_delta(total_assets, 4), subindustry)
Submitting: group_rank(cash_from_operations / (net_income + 1e-6), subindustry)
  sim_id=4uMUzO2PM4olccfuu5lJz0n — polling every 10s...
  status=ERROR
  sharpe=None  fitness=None  group_rank(cash_from_operations / (net_income + 1e-6), subin
Submitting: group_rank(assets, industry)
  sim_id=1m02313AF4Fo9qf16SUwN7iy — polling every 10s...
  status=COMPLETE
  sharpe=0.85  fitness=0.67  group

## Results

In [9]:
all_results = load_results()
fund_results = all_results[all_results['tags'].str.contains('fundamental', na=False)].copy()
fund_results = fund_results.dropna(subset=['is_sharpe'])

print(f'{len(fund_results)} fundamental results')
top_alphas(fund_results, n=15)[['expression', 'is_sharpe', 'is_fitness', 'is_returns', 'is_turnover', 'decay']]

23 fundamental results


,expression,is_sharpe,is_fitness,is_returns,is_turnover,decay
0,"group_rank(ts_delta(assets, 4), subindustry)",1.79,1.34,0.1168,0.2077,10
1,"group_rank(ts_delta(liabilities, 4), subindustry)",1.69,1.22,0.1081,0.2060,10
2,"group_rank(ts_delta(operating_income, 4), subi...",1.33,0.83,0.0793,0.2028,10
3,"group_rank(liabilities, industry)",0.99,0.82,0.0866,0.0165,10
4,"group_rank(liabilities, subindustry)",0.99,0.78,0.0783,0.0152,10
5,"group_rank(sales, industry)",0.87,0.72,0.0853,0.0140,10
6,rank(liabilities),0.82,0.70,0.0903,0.0138,10
7,"group_rank(assets, industry)",0.85,0.67,0.0773,0.0162,10
8,rank(sales),0.74,0.63,0.0900,0.0112,10
9,"group_rank(assets, subindustry)",0.82,0.60,0.0679,0.0150,10
